# InsideForest: caso de uso reproducible

Este notebook es la versión canónica incluida en la propia librería. Presenta:

1. el flujo tradicional de **clustering supervisado** con Iris;
2. utilidades que suelen quedar fuera de una demostración corta, como importancias y persistencia;
3. un flujo de **clasificación multiclase real** con Wine (tres clases);
4. reglas prototipo, regiones de confusión, asignación, margen y fallback;
5. un ejemplo breve de regresión.

Todos los datasets vienen con scikit-learn. No se necesita una API key ni descargar archivos de datos.


In [ ]:
# En Colab instala InsideForest solamente cuando todavía no está disponible.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("InsideForest") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "InsideForest>=0.4.0"]
    )


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split

from InsideForest import (
    InsideForestClassifier,
    InsideForestRegressor,
)
from InsideForest.multiclass import (
    InsideForestMulticlassClassifier,
    extract_multiclass_leaf_rules,
    get_multiclass_labels,
)

try:
    insideforest_version = version("InsideForest")
except PackageNotFoundError:
    insideforest_version = "código fuente local"

print("InsideForest:", insideforest_version)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 20)


# 1. Flujo tradicional: clustering supervisado con Iris

`InsideForestClassifier` usa el objetivo para descubrir regiones y devuelve **etiquetas de cluster**.
Estas etiquetas no son IDs de clase y por eso se analizan como segmentos supervisados.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X_iris = iris.data.copy()
X_iris.columns = [
    column.replace(" (cm)", "").replace(" ", "_")
    for column in X_iris.columns
]
y_iris = iris.target.to_numpy()

iris_frame = X_iris.copy()
iris_frame["species"] = y_iris
iris_frame.head()


In [ ]:
traditional = InsideForestClassifier(
    rf_params={
        "n_estimators": 16,
        "max_depth": 5,
        "random_state": 15,
        "n_jobs": 1,
    },
    tree_params={
        "lang": "py",
        "n_sample_multiplier": 0.05,
        "ef_sample_multiplier": 10,
    },
    var_obj="species",
    no_trees_search=16,
    leaf_percentile=95,
    low_leaf_fraction=0.10,
    max_cases=len(X_iris),
    get_detail=True,
    seed=15,
)

traditional.fit(iris_frame)
iris_cluster_labels = traditional.predict(X_iris)

pd.Series(
    {
        "rf_accuracy": traditional.score(X_iris, y_iris),
        "n_clusters": pd.Series(iris_cluster_labels).nunique(),
        "unmatched_rate": np.mean(iris_cluster_labels == -1),
        "n_described_regions": len(traditional.df_clusters_description_),
    },
    name="Iris",
)


In [ ]:
cluster_table = (
    pd.DataFrame({"species": y_iris, "cluster": iris_cluster_labels})
    .groupby(["species", "cluster"])
    .size()
    .unstack(fill_value=0)
)

ax = cluster_table.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    colormap="viridis",
)
ax.set(
    title="Iris: clase real frente a cluster supervisado",
    xlabel="Clase real",
    ylabel="Observaciones",
)
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
traditional.df_clusters_description_.sort_values(
    ["cluster_weight", "cluster_n_sample"],
    ascending=False,
).head(10)


In [ ]:
frontiers = traditional.frontiers_

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    frontiers["delta_cluster_ef_sample"],
    frontiers["similarity"],
    c=frontiers["score"],
    cmap="viridis",
    alpha=0.7,
)
ax.set(
    title="Comparación entre clusters",
    xlabel="Diferencia de efectividad",
    ylabel="Similitud",
)
ax.grid(linestyle="--", alpha=0.4)
fig.colorbar(scatter, ax=ax, label="Score")
plt.tight_layout()
plt.show()


## Importancias y persistencia

El wrapper expone las importancias del bosque y puede guardarse/cargarse sin volver a entrenar.


In [ ]:
importance_table = (
    pd.DataFrame(
        {
            "feature": traditional.feature_names_,
            "importance": traditional.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

ax = importance_table.plot.bar(
    x="feature",
    y="importance",
    legend=False,
    figsize=(8, 4),
    color="#4C78A8",
)
ax.set(title="Importancias del Random Forest", ylabel="Importancia")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

importance_table


In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    model_path = Path(tmp_dir) / "insideforest_iris.joblib"
    traditional.save(model_path)
    restored = InsideForestClassifier.load(model_path)
    restored_labels = restored.predict(X_iris)

print(
    "Predicciones idénticas después de guardar/cargar:",
    np.array_equal(iris_cluster_labels, restored_labels),
)


# 2. Clasificación con más de dos clases: Wine

Aquí sí usamos `InsideForestMulticlassClassifier`. A diferencia del flujo tradicional:

- cada hoja conserva la distribución completa de probabilidad por clase;
- las reglas se puntúan desde la perspectiva de cada clase;
- la salida es una clase predicha, con confianza, segunda clase y margen;
- si ninguna región seleccionada cubre una fila, se usa el bosque como fallback explícito.

Wine tiene tres clases y trece variables numéricas.


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine = wine.data.copy()
y_wine = np.asarray(
    [wine.target_names[index] for index in wine.target],
    dtype=object,
)

X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine,
    y_wine,
    test_size=0.25,
    stratify=y_wine,
    random_state=42,
)

pd.Series(y_wine_train).value_counts().sort_index()


In [ ]:
multiclass_model = InsideForestMulticlassClassifier(
    rf_params={
        "n_estimators": 24,
        "max_depth": 4,
        "min_samples_leaf": 4,
        "random_state": 42,
        "n_jobs": 1,
    },
    percentil=90,
    low_frac=0.20,
    min_support=3,
    max_rules_per_class=60,
    random_state=42,
    n_jobs=1,
    conflict_margin=0.35,
)

multiclass_model.fit(X_wine_train, y_wine_train)

pd.Series(
    {
        "classes": list(multiclass_model.classes_),
        "n_rules": len(multiclass_model.rules_),
        "classes_in_rules": multiclass_model.rules_["target_class"].nunique(),
    },
    name="Wine",
)


## Reglas por clase

Cada regla contiene pureza, cobertura, lift y un vector completo `class_distribution`.
El `target_class` indica desde qué clase se está evaluando esa hoja.


In [ ]:
def format_distribution(probabilities, classes):
    return {
        str(label): round(float(probability), 3)
        for label, probability in zip(classes, probabilities)
    }


top_rules = multiclass_model.explain(top_n=12).copy()
top_rules["class_distribution"] = top_rules["class_distribution"].map(
    lambda values: format_distribution(values, multiclass_model.classes_)
)

top_rules[
    [
        "target_class",
        "support",
        "coverage",
        "target_probability",
        "lift",
        "entropy",
        "score",
        "class_distribution",
        "description",
    ]
]


In [ ]:
label_report = multiclass_model.labels_.copy()
label_report["class_distribution"] = label_report["class_distribution"].map(
    lambda values: format_distribution(values, multiclass_model.classes_)
)

label_report.sort_values(
    ["target_probability", "support"],
    ascending=False,
).head(10)


## Asignación sobre datos no vistos

`assign_regions` informa si la asignación vino de una región o del fallback del bosque.


In [ ]:
wine_assignments = multiclass_model.assign_regions(X_wine_test)
wine_results = wine_assignments.assign(actual_class=y_wine_test)

metrics = pd.Series(
    {
        "accuracy": accuracy_score(
            wine_results["actual_class"],
            wine_results["predicted_class"],
        ),
        "fallback_rate": np.mean(
            wine_results["source"] == "model_fallback"
        ),
        "conflict_rate": wine_results["is_conflict"].mean(),
        "mean_confidence": wine_results["confidence"].mean(),
        "mean_margin": wine_results["margin"].mean(),
    },
    name="holdout",
)

display(metrics)
wine_results.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    wine_results["actual_class"],
    wine_results["predicted_class"],
    labels=list(multiclass_model.classes_),
    xticks_rotation=35,
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Wine: matriz de confusión")

source_colors = wine_results["source"].map(
    {"region": "#2E86AB", "model_fallback": "#F18F01"}
)
axes[1].scatter(
    wine_results["confidence"],
    wine_results["margin"],
    c=source_colors,
    alpha=0.75,
)
axes[1].axhline(
    multiclass_model.conflict_margin,
    color="crimson",
    linestyle="--",
    label="umbral de conflicto",
)
axes[1].set(
    title="Confianza frente a margen top-2",
    xlabel="Confianza",
    ylabel="Margen",
)
axes[1].legend()
axes[1].grid(linestyle="--", alpha=0.35)

plt.tight_layout()
plt.show()


## Regiones prototipo y regiones de confusión

Los prototipos son las reglas mejor puntuadas por clase. Las regiones de confusión son hojas donde
las dos probabilidades más altas están cerca.


In [ ]:
prototypes = multiclass_model.prototype_regions(top_n=3).copy()
prototype_view = prototypes[
    [
        "target_class",
        "support",
        "target_probability",
        "lift",
        "score",
        "description",
    ]
].sort_values(["target_class", "score"], ascending=[True, False])

display(prototype_view)

plot_prototypes = prototype_view.copy()
plot_prototypes["prototype"] = (
    plot_prototypes["target_class"].astype(str)
    + " #"
    + (plot_prototypes.groupby("target_class").cumcount() + 1).astype(str)
)
ax = plot_prototypes.plot.barh(
    x="prototype",
    y="score",
    legend=False,
    figsize=(9, 5),
    color="#5B8E7D",
)
ax.invert_yaxis()
ax.set(title="Score de regiones prototipo", xlabel="Score", ylabel="")
plt.tight_layout()
plt.show()


In [ ]:
confusion_regions = multiclass_model.confusion_regions(top_n=10)

if confusion_regions.empty:
    print("No se encontraron regiones de confusión con el umbral actual.")
else:
    display(
        confusion_regions[
            [
                "top_class",
                "top_probability",
                "second_class",
                "second_probability",
                "margin",
                "support",
                "description",
            ]
        ]
    )


## Fallback explícito

En una ejecución normal el fallback aparece cuando ninguna de las regiones seleccionadas cubre una fila.
Para mostrar la ruta de forma determinista, la siguiente celda desactiva temporalmente las reglas,
asigna dos filas y luego restaura el modelo.


In [ ]:
selected_rules = multiclass_model.rules_
try:
    multiclass_model.rules_ = selected_rules.iloc[0:0].copy()
    fallback_demo = multiclass_model.assign_regions(X_wine_test.head(2))
finally:
    multiclass_model.rules_ = selected_rules

fallback_demo


## API de bajo nivel

La fachada `InsideForest.multiclass` también permite extraer reglas y recalcular sus etiquetas
cuando se necesita construir un flujo personalizado.


In [ ]:
custom_rules = extract_multiclass_leaf_rules(
    multiclass_model.rf_,
    X_wine_train,
    y_wine_train,
    percentil=95,
    low_frac=0.05,
    min_support=3,
    max_rules_per_class=10,
    random_state=42,
)

custom_labels = get_multiclass_labels(
    custom_rules,
    X_wine_train,
    y_wine_train,
    class_labels=multiclass_model.classes_,
)

custom_labels[
    [
        "target_class",
        "support",
        "coverage",
        "target_probability",
        "lift",
        "entropy",
        "description",
    ]
].sort_values(["target_class", "target_probability"], ascending=[True, False]).head(12)


# 3. Flujo breve de regresión

`InsideForestRegressor` conserva el mismo patrón de uso. `score` reporta el R² del bosque,
mientras que `predict` devuelve las regiones interpretables asignadas.


In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    diabetes.data,
    diabetes.target,
    test_size=0.25,
    random_state=42,
)

regressor = InsideForestRegressor(
    rf_params={
        "n_estimators": 12,
        "max_depth": 5,
        "random_state": 42,
        "n_jobs": 1,
    },
    no_trees_search=12,
    max_cases=len(X_reg_train),
    get_detail=False,
    seed=42,
)
regressor.fit(X_reg_train, y_reg_train)

regression_regions = regressor.predict(X_reg_test)
pd.Series(
    {
        "rf_r2": regressor.score(X_reg_test, y_reg_test),
        "assigned_regions": pd.Series(regression_regions).nunique(),
        "unmatched_rate": np.mean(regression_regions == -1),
    },
    name="Diabetes",
)


# Siguientes pasos

- Usa `InsideForestClassifier` cuando el resultado que buscas son segmentos o clusters supervisados.
- Usa `InsideForestMulticlassClassifier` cuando necesitas reglas interpretables que produzcan clases,
  probabilidades, margen y fallback para objetivos nominales de dos o más clases.
- Para validación comparativa y benchmarks, revisa `README.multiclass.md` y `experiments/`.
